#**RFdiffusion**
RFdiffusion is a method for structure generation, with or without conditional information (a motif, target etc). It can perform a whole range of protein design challenges as we have outlined in the RFdiffusion [manuscript](https://www.biorxiv.org/content/10.1101/2022.12.09.519842v2).

**<font color="red">NOTE:</font>** This notebook is in development, we are still working on adding all the options from the manuscript above.

For **instructions**, see end of Notebook.

See [diffusion_foldcond](https://colab.research.google.com/github/sokrypton/ColabDesign/blob/main/rf/examples/diffusion_foldcond.ipynb) for fold conditioning functionality.

See [original version](https://colab.research.google.com/github/sokrypton/ColabDesign/blob/main/rf/examples/diffusion_ori.ipynb) of this notebook (from 31Mar2023).



In [0]:
%pip install -q git+https://github.com/KrajTechne/ColabDesign --no-deps

In [0]:
%restart_python

In [0]:
import sys
import os
sys.path.append("/Workspace/Users/karthik.raj@bio-techne.com/RFdiffusion")
sys.path.append("/Workspace/Users/karthik.raj@bio-techne.com/ColabDesign")
os.chdir("/Workspace/Users/karthik.raj@bio-techne.com/RFdiffusion")

In [0]:
import os, time, signal
import sys, random, string, re
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import ipywidgets as widgets
import py3Dmol
import subprocess

from inference.utils import parse_pdb
from colabdesign.rf.utils import get_ca
from colabdesign.rf.utils import fix_contigs, fix_partial_contigs, fix_pdb, sym_it
from colabdesign.shared.protein import pdb_to_string
from colabdesign.shared.plot import plot_pseudo_3D

import warnings
warnings.filterwarnings("ignore")

# Disable autoreload to prevent AutoreloadReliabilityHook callback errors
try:
    ip = get_ipython()
    ip.magics_manager.registry.pop('AutoreloadMagics', None)
    # Remove the post_run_cell callback that causes the error
    ip.events.callbacks['post_run_cell'] = [
        cb for cb in ip.events.callbacks.get('post_run_cell', [])
        if 'AutoreloadReliabilityHook' not in str(cb)
    ]
except Exception:
    pass

def get_pdb(pdb_code=None):
  if pdb_code is None or pdb_code == "":
    raise ValueError("Please provide path to a PDB file")
  elif os.path.isfile(pdb_code):
    return pdb_code
  elif len(pdb_code) == 4:
    if not os.path.isfile(f"{pdb_code}.pdb1"):
      os.system(f"wget -qnc https://files.rcsb.org/download/{pdb_code}.pdb1.gz")
      os.system(f"gunzip {pdb_code}.pdb1.gz")
    return f"{pdb_code}.pdb1"
  else:
    os.system(f"wget -qnc https://alphafold.ebi.ac.uk/files/AF-{pdb_code}-F1-model_v3.pdb")
    return f"AF-{pdb_code}-F1-model_v3.pdb"

def run_ananas(pdb_str, path, sym=None):
  pdb_filename = f"outputs/{path}/ananas_input.pdb"
  out_filename = f"outputs/{path}/ananas.json"
  with open(pdb_filename,"w") as handle:
    handle.write(pdb_str)

  cmd = f"./ananas {pdb_filename} -u -j {out_filename}"
  if sym is None: os.system(cmd)
  else: os.system(f"{cmd} {sym}")

  try:
    out = json.loads(open(out_filename,"r").read())
    results,AU = out[0], out[-1]["AU"]
    group = AU["group"]
    chains = AU["chain names"]
    rmsd = results["Average_RMSD"]
    print(f"AnAnaS detected {group} symmetry at RMSD:{rmsd:.3}")

    C = np.array(results['transforms'][0]['CENTER'])
    A = [np.array(t["AXIS"]) for t in results['transforms']]

    new_lines = []
    for line in pdb_str.split("\n"):
      if line.startswith("ATOM"):
        chain = line[21:22]
        if chain in chains:
          x = np.array([float(line[i:(i+8)]) for i in [30,38,46]])
          if group[0] == "c":
            x = sym_it(x,C,A[0])
          if group[0] == "d":
            x = sym_it(x,C,A[1],A[0])
          coord_str = "".join(["{:8.3f}".format(a) for a in x])
          new_lines.append(line[:30]+coord_str+line[54:])
      else:
        new_lines.append(line)
    return results, "\n".join(new_lines)

  except:
    return None, pdb_str


def run(command_list, steps, num_designs=1, visual="none"):
  
  run_output = widgets.Output()
  progress = widgets.FloatProgress(min=0, max=1, description='running', bar_style='info')
  display(widgets.VBox([progress, run_output]))

  # clear previous run
  for n in range(steps):
    if os.path.isfile(f"/dev/shm/{n}.pdb"):
      os.remove(f"/dev/shm/{n}.pdb")

  # --- NEW: Use Popen to run the script non-blocking in the background ---
  print("Executing command:", " ".join(command_list))
  process = subprocess.Popen(command_list)
  
  try:
    fail = False
    for _ in range(num_designs):

      # for each step check if output generated
      for n in range(steps):
        wait = True
        while wait and not fail:
          time.sleep(0.1)
          if os.path.isfile(f"/dev/shm/{n}.pdb"):
            pdb_str = open(f"/dev/shm/{n}.pdb").read()
            if pdb_str[-3:] == "TER":
              wait = False
            # process.poll() returns None if it is still running
            elif process.poll() is not None:
              fail = True
          elif process.poll() is not None:
            fail = True

        if fail:
          progress.bar_style = 'danger'
          progress.description = "failed"
          break

        else:
          progress.value = (n+1) / steps
          if visual != "none":
            with run_output:
              run_output.clear_output(wait=True)
              if visual == "image":
                xyz, bfact = get_ca(f"/dev/shm/{n}.pdb", get_bfact=True)
                fig = plt.figure()
                fig.set_dpi(100);fig.set_figwidth(6);fig.set_figheight(6)
                ax1 = fig.add_subplot(111);ax1.set_xticks([]);ax1.set_yticks([])
                plot_pseudo_3D(xyz, c=bfact, cmin=0.5, cmax=0.9, ax=ax1)
                plt.show()
              if visual == "interactive":
                view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js')
                view.addModel(pdb_str,'pdb')
                view.setStyle({'cartoon': {'colorscheme': {'prop':'b','gradient': 'roygb','min':0.5,'max':0.9}}})
                view.zoomTo()
                view.show()
        if os.path.exists(f"/dev/shm/{n}.pdb"):
          os.remove(f"/dev/shm/{n}.pdb")
      if fail:
        progress.bar_style = 'danger'
        progress.description = "failed"
        break

    # Wait for the background script to finish naturally
    while process.poll() is None:
      time.sleep(0.1)

  except KeyboardInterrupt:
    process.terminate()
    progress.bar_style = 'danger'
    progress.description = "stopped"

def run_diffusion(contigs, path, pdb=None, iterations=50,
                  symmetry="none", order=1, hotspot=None,
                  chains=None, add_potential=False, partial_T="auto",
                  num_designs=1, use_beta_model=False, visual="none"):

  full_path = f"outputs/{path}" #--------------------------------------- modify to be the temp directory
  os.makedirs(full_path, exist_ok=True)
  opts = [f"inference.output_prefix={full_path}",
          f"inference.num_designs={num_designs}"]

  if chains == "": chains = None

  if symmetry in ["auto","cyclic","dihedral"]:
    if symmetry == "auto":
      sym, copies = None, 1
    else:
      sym, copies = {"cyclic":(f"c{order}",order),
                     "dihedral":(f"d{order}",order*2)}[symmetry]
  else:
    symmetry = None
    sym, copies = None, 1

  contigs = contigs.replace(","," ").replace(":"," ").split()
  is_fixed, is_free = False, False
  fixed_chains = []
  for contig in contigs:
    for x in contig.split("/"):
      a = x.split("-")[0]
      if a[0].isalpha():
        is_fixed = True
        if a[0] not in fixed_chains:
          fixed_chains.append(a[0])
      if a.isnumeric():
        is_free = True
  if len(contigs) == 0 or not is_free:
    mode = "partial"
  elif is_fixed:
    mode = "fixed"
  else:
    mode = "free"

  if mode in ["partial","fixed"]:
    pdb_str = pdb_to_string(get_pdb(pdb), chains=chains)
    if symmetry == "auto":
      a, pdb_str = run_ananas(pdb_str, path)
      if a is None:
        print(f'ERROR: no symmetry detected')
        symmetry = None
        sym, copies = None, 1
      else:
        if a["group"][0] == "c":
          symmetry = "cyclic"
          sym, copies = a["group"], int(a["group"][1:])
        elif a["group"][0] == "d":
          symmetry = "dihedral"
          sym, copies = a["group"], 2 * int(a["group"][1:])
        else:
          print(f'ERROR: the detected symmetry ({a["group"]}) not currently supported')
          symmetry = None
          sym, copies = None, 1

    elif mode == "fixed":
      pdb_str = pdb_to_string(pdb_str, chains=fixed_chains)

    pdb_filename = f"{full_path}/input.pdb"
    with open(pdb_filename, "w") as handle:
      handle.write(pdb_str)

    parsed_pdb = parse_pdb(pdb_filename)
    opts.append(f"inference.input_pdb={pdb_filename}")
    if mode in ["partial"]:
      if partial_T == "auto":
        iterations = int(80 * (iterations / 200))
      else:
        iterations = int(partial_T)
      opts.append(f"diffuser.partial_T={iterations}")
      contigs = fix_partial_contigs(contigs, parsed_pdb)
    else:
      opts.append(f"diffuser.T={iterations}")
      contigs = fix_contigs(contigs, parsed_pdb)
  else:
    opts.append(f"diffuser.T={iterations}")
    parsed_pdb = None
    contigs = fix_contigs(contigs, parsed_pdb)

  if hotspot is not None and hotspot != "":
    hotspot = ",".join(hotspot.replace(","," ").split())
    opts.append(f"ppi.hotspot_res=[{hotspot}]")

  if sym is not None:
    sym_opts = ["--config-name", "symmetry", f"inference.symmetry={sym}"]
    if add_potential:
      sym_opts += ["potentials.guiding_potentials=[\"type:olig_contacts,weight_intra:1,weight_inter:0.1\"]",
                   "potentials.olig_intra_all=True","potentials.olig_inter_all=True",
                   "potentials.guide_scale=2","potentials.guide_decay=quadratic"]
    opts = sym_opts + opts
    contigs = sum([contigs] * copies,[])

  opts.append(f"contigmap.contigs=[{' '.join(contigs)}]")
  opts += ["inference.dump_pdb=True", "inference.dump_pdb_path=/dev/shm"]
  if use_beta_model:
    opts += ["inference.ckpt_override_path= /Volumes/sandbox/model_weights/protein_hunter/models_rfdiffusion/Complex_beta_ckpt.pt"]

  print("mode:", mode)
  print("output:", full_path)
  print("contigs:", contigs)

  # --- Build the final command list for Popen ---
  cmd = ["python", "run_inference.py", "--config-name", "base"] + opts
  
  # RUN
  run(cmd, iterations, num_designs, visual=visual)

  # fix pdbs
  for n in range(num_designs):
    pdbs = [f"outputs/traj/{path}_{n}_pX0_traj.pdb",
            f"outputs/traj/{path}_{n}_Xt-1_traj.pdb",
            f"{full_path}_{n}.pdb"]
    for pdb in pdbs:
      if os.path.exists(pdb):
        with open(pdb,"r") as handle: pdb_str = handle.read()
        with open(pdb,"w") as handle: handle.write(fix_pdb(pdb_str, contigs))

  return contigs, copies

In [0]:
import yaml
dbutils.widgets.text('yaml_filename', 'base.yaml')
yaml_filename = dbutils.widgets.get('yaml_filename')
path_design_campaign = "/Volumes/sandbox/karthik/motif_scaffolding/alpha2_beta1_double_globular_trial3"
yaml_folder = f"{path_design_campaign}/rfdiffusion/yaml/"
yaml_path = yaml_folder + yaml_filename
with open(yaml_path, 'r') as handle:
  rfd_config = yaml.safe_load(handle)

In [0]:
print(yaml_filename)
print("Full Configuration for RFdiffusion, AlphaFold2, and ProteinMPNN")
print(rfd_config)

In [0]:
#@title run **RFdiffusion** to generate a backbone
name = yaml_filename.split('.')[0] # globular_double_strand_test

contigs = rfd_config['rfdiffusion']['contigs']
#contigs = "A:20-35/B5-13/40-60/C5-13/3-8"
pdb = rfd_config['rfdiffusion']['pdb'] 
iterations = rfd_config['rfdiffusion']['iterations'] # ["25", "50", "100", "150", "200"] 
hotspot = rfd_config['rfdiffusion']['hotspot']  # {type:"string"}
num_designs = rfd_config['rfdiffusion']['num_designs']  # ["1", "2", "4", "8", "16", "32"] {type:"raw"} 
visual = rfd_config['rfdiffusion']['visual'] #@param ["none", "image", "interactive"]


#@markdown **symmetry** settings
symmetry = rfd_config['rfdiffusion']['symmetry'] #@param ["none", "auto", "cyclic", "dihedral"]
order = rfd_config['rfdiffusion']['order'] #@param ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12"] {type:"raw"}
chains = rfd_config['rfdiffusion']['chains'] #@param {type:"string"}
add_potential = rfd_config['rfdiffusion']['add_potential'] # {type: boolean} = True

#@markdown - specify number of noising steps (only used for the partial diffusion protocol)
partial_T = rfd_config['rfdiffusion']['partial_T'] # @param ["auto", "10", "20", "40", "60", "80"]
#@markdown - if you are seeing lots of helices, switch to the "beta" params for a better SSE balance.
use_beta_model = rfd_config['rfdiffusion']['use_beta_model'] #@param {type:"boolean"}


# determine where to save
path = name
while os.path.exists(f"outputs/{path}_0.pdb"):
  path = name + "_" + ''.join(random.choices(string.ascii_lowercase + string.digits, k=5))

flags = {"contigs":contigs,
         "pdb":pdb,
         "order":order,
         "iterations":iterations,
         "symmetry":symmetry,
         "hotspot":hotspot,
         "path":path,
         "chains":chains,
         "add_potential":add_potential,
         "num_designs":num_designs,
         "use_beta_model":use_beta_model,
         "visual":visual,
         "partial_T":partial_T}

for k,v in flags.items():
  if isinstance(v,str):
    flags[k] = v.replace("'","").replace('"','')

contigs, copies = run_diffusion(**flags)

In [0]:
#@title Display 3D structure {run: "auto"}
animate = "none" #@param ["none", "movie", "interactive"]
color = "chain" #@param ["rainbow", "chain", "plddt"]
denoise = True
dpi = 100 #@param ["100", "200", "400"] {type:"raw"}
from colabdesign.shared.plot import pymol_color_list
from colabdesign.rf.utils import get_ca, get_Ls, make_animation
from string import ascii_uppercase,ascii_lowercase
alphabet_list = list(ascii_uppercase+ascii_lowercase)

def plot_pdb(num=0):
  if denoise:
    pdb_traj = f"outputs/traj/{path}_{num}_pX0_traj.pdb"
  else:
    pdb_traj = f"outputs/traj/{path}_{num}_Xt-1_traj.pdb"
  if animate in ["none","interactive"]:
    hbondCutoff = 4.0
    view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js')
    if animate == "interactive":
      pdb_str = open(pdb_traj,'r').read()
      view.addModelsAsFrames(pdb_str,'pdb',{'hbondCutoff':hbondCutoff})
    else:
      pdb = f"outputs/{path}_{num}.pdb"
      pdb_str = open(pdb,'r').read()
      view.addModel(pdb_str,'pdb',{'hbondCutoff':hbondCutoff})
    if color == "rainbow":
      view.setStyle({'cartoon': {'color':'spectrum'}})
    elif color == "chain":
      for n,chain,c in zip(range(len(contigs)),
                              alphabet_list,
                              pymol_color_list):
          view.setStyle({'chain':chain},{'cartoon': {'color':c}})
    else:
      view.setStyle({'cartoon': {'colorscheme': {'prop':'b','gradient': 'roygb','min':0.5,'max':0.9}}})
    view.zoomTo()
    if animate == "interactive":
      view.animate({'loop': 'backAndForth'})
    view.show()
  else:
    Ls = get_Ls(contigs)
    xyz, bfact = get_ca(pdb_traj, get_bfact=True)
    xyz = xyz.reshape((-1,sum(Ls),3))[::-1]
    bfact = bfact.reshape((-1,sum(Ls)))[::-1]
    if color == "chain":
      display(HTML(make_animation(xyz, Ls=Ls, dpi=dpi, ref=-1)))
    elif color == "rainbow":
      display(HTML(make_animation(xyz, dpi=dpi, ref=-1)))
    else:
      display(HTML(make_animation(xyz, plddt=bfact*100, dpi=dpi, ref=-1)))


if num_designs > 1:
  output = widgets.Output()
  def on_change(change):
    if change['name'] == 'value':
      with output:
        output.clear_output(wait=True)
        plot_pdb(change['new'])
  dropdown = widgets.Dropdown(
      options=[(f'{k}',k) for k in range(num_designs)],
      value=0, description='design:',
  )
  dropdown.observe(on_change)
  display(widgets.VBox([dropdown, output]))
  with output:
    plot_pdb(dropdown.value)
else:
  plot_pdb()

In [0]:
#@title run **ProteinMPNN** to generate a sequence and **AlphaFold** to validate
#@markdown ProteinMPNN Settings
import subprocess
num_seqs = rfd_config['mpnn']['num_seqs'] #@param ["1", "2", "4", "8", "16", "32", "64"] {type:"raw"}

#@markdown - `mpnn_sampling_temp` - control diversity of sampled sequences. (higher = more diverse).
#@markdown - `rm_aa='C'` - do not use [C]ysteines.
#@markdown - `use_solubleMPNN` - Weights trained only on soluble proteins. (https://www.biorxiv.org/content/10.1101/2023.05.09.540044v2).
mpnn_sampling_temp = rfd_config['mpnn']['mpnn_sampling_temp'] #@param ["0.0001", "0.1", "0.15", "0.2", "0.25", "0.3", "0.5", "1.0"]
rm_aa = rfd_config['mpnn']['rm_aa'] #@param {type:"string"}
use_solubleMPNN = rfd_config['mpnn']['use_solubleMPNN'] #@param {type:"boolean"}

#@markdown
#@markdown AlphaFold Settings
initial_guess = rfd_config['alphafold2']['initial_guess'] #@param {type:"boolean"}
#@markdown - soft initialization with desired coordinates, see [paper](https://www.nature.com/articles/s41467-023-38328-5).
num_recycles = rfd_config['alphafold2']['num_recycles'] #@param ["0", "1", "2", "3", "6", "12"] {type:"raw"}
#@markdown - for **binder** design, we recommend `initial_guess=True num_recycles=3`
use_multimer = rfd_config['alphafold2']['use_multimer'] #@param {type:"boolean"}
#@markdown - `use_multimer` - use AlphaFold Multimer v3 params for prediction.

contigs_str = ":".join(contigs)
opts = [f"--pdb=outputs/{path}_0.pdb",
        f"--loc=outputs/{path}",
        f"--contig={contigs_str}",
        f"--copies={copies}",
        f"--num_seqs={num_seqs}",
        f"--num_recycles={num_recycles}",
        f"--rm_aa={rm_aa}",
        f"--mpnn_sampling_temp={mpnn_sampling_temp}",
        f"--num_designs={num_designs}"]
if initial_guess: opts.append("--initial_guess")
if use_multimer: opts.append("--use_multimer")
if use_solubleMPNN: opts.append("--use_soluble")

# Add in af_params_dir path:
af_params_dir = "/Volumes/sandbox/model_weights/protein_hunter/params/"
opts += [f"--af_params_dir={af_params_dir}"]

cmd = ['python', '/Workspace/Users/karthik.raj@bio-techne.com/ColabDesign/colabdesign/rf/designability_test.py'] + opts
print(" ".join(cmd))
subprocess.run(cmd)

In [0]:
#@title Display best result
import py3Dmol
def plot_pdb(num = "best"):
  if num == "best":
    with open(f"outputs/{path}/best.pdb","r") as f:
      # REMARK 001 design {m} N {n} RMSD {rmsd}
      info = f.readline().strip('\n').split()
    num = info[3]
  hbondCutoff = 4.0
  view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js')
  pdb_str = open(f"outputs/{path}_{num}.pdb",'r').read()
  view.addModel(pdb_str,'pdb',{'hbondCutoff':hbondCutoff})
  pdb_str = open(f"outputs/{path}/best_design{num}.pdb",'r').read()
  view.addModel(pdb_str,'pdb',{'hbondCutoff':hbondCutoff})

  view.setStyle({"model":0},{'cartoon':{}}) #: {'colorscheme': {'prop':'b','gradient': 'roygb','min':0,'max':100}}})
  view.setStyle({"model":1},{'cartoon':{'colorscheme': {'prop':'b','gradient': 'roygb','min':0,'max':100}}})
  view.zoomTo()
  view.show()

if num_designs > 1:
  def on_change(change):
    if change['name'] == 'value':
      with output:
        output.clear_output(wait=True)
        plot_pdb(change['new'])
  dropdown = widgets.Dropdown(
    options=["best"] + [str(k) for k in range(num_designs)],
    value="best",
    description='design:',
  )
  dropdown.observe(on_change)
  output = widgets.Output()
  display(widgets.VBox([dropdown, output]))
  with output:
    plot_pdb(dropdown.value)
else:
  plot_pdb()

In [0]:
import sys
sys.path.append('/Workspace/Users/karthik.raj@bio-techne.com/ScoringPhysics')
from StrucTools import visualize_structure
design_id = 2
pdb_path = f"outputs/{path}/best_design{design_id}.pdb"
visualize_structure(pdb_path)

Computing RMSD of the Binder Chain from RFDiffusion to RMSD of Binder Chain from MPNN derived seq whose structure predicted by AF2

In [0]:
import pandas as pd
import os
path_mpnn_csv = f"outputs/{path}/mpnn_results.csv"
df_mpnn = pd.read_csv(path_mpnn_csv, index_col=0)
df_mpnn['contig'] = ';'.join(contigs)
df_mpnn

In [0]:
import sys
sys.path.append('/Workspace/Users/karthik.raj@bio-techne.com/ScoringPhysics')
from StrucTools import *
import biotite.structure as struc
path_pdbs = f"outputs/{path}/all_pdb"
list_binder_rfd_af2_holo_rmsd = []
for index, row in df_mpnn.iterrows():
    # 1. Define PDB paths of holo complexes for RFDiffusion input to MPNN and the MPNN-derived seqs whose structures predicted via AF2
    pdb_af2_path = path_pdbs + f"/design{row['design']}_n{row['n']}.pdb"
    pdb_rfd_path = f"outputs/{path}_{row['design']}.pdb"
    # 2. Extract only the alpha-carbons from the holo complexes
    atom_array_complex_rfd = extract_atom_array(pdb_rfd_path, ca_only = True)
    atom_array_complex_af2 = extract_atom_array(pdb_af2_path, ca_only = True)
    # 3. Create mask for only the binder chain residues
    residue_binder_mask = atom_array_complex_rfd.chain_id == 'B'
    # 4. Spread mask to all atoms in the entire complex
    atom_binder_mask = struc.spread_residue_wise(atom_array_complex_rfd, residue_binder_mask)
    # 5. Superimpose the 2 structures at the atom_binder_mask and calculate the RMSD for the binder chains
    rmsd, _, _ = superimpose_and_calculate_specified_rmsd(atom_array_complex_rfd, atom_array_complex_af2, atom_binder_mask, atom_binder_mask)
    print("RMSD between MPNN-seq AF2 predicted holo structure and original RFDiffusion Holo structure: ", rmsd)
    list_binder_rfd_af2_holo_rmsd.append(rmsd)
df_mpnn['rmsd_holo_binder_rfd_af2'] = list_binder_rfd_af2_holo_rmsd
df_mpnn.to_csv(path_mpnn_csv)
df_mpnn

In [0]:
import shutil
current_save_path = os.path.join(os.getcwd(), "outputs", path)
volume_save_path = f"{path_design_campaign}/rfdiffusion/designs/{path}"
shutil.copytree(current_save_path, volume_save_path, dirs_exist_ok=True)

**Instructions**
---
---

Use `contigs` to define continious chains. Use a `:` to define multiple contigs and a `/` to define mutliple segments within a contig.
For example:

**unconditional**
- `contigs='100'` - diffuse **monomer** of length 100
- `contigs='50:100'` - diffuse **hetero-oligomer** of lengths 50 and 100
- `contigs='50'` `symmetry='cyclic'` `order=2` - make two copies of the defined contig(s) and add a symmetry constraint, for **homo-oligomeric** diffusion.

**binder design**
- `contigs='A:50'` `pdb='4N5T'` - diffuse a **binder** of length 50 to chain A of defined PDB.
- `contigs='E6-155:70-100'` `pdb='5KQV'` `hotspot='E64,E88,E96'` - diffuse a **binder** of length 70 to 100 (sampled randomly) to chain E and defined hotspot(s).

**motif scaffolding**
 - `contigs='40/A163-181/40'` `pdb='5TPN'`
 - `contigs='A3-30/36/A33-68'` `pdb='6MRR'` - diffuse a loop of length 36 between two segments of defined PDB ranges.

**partial diffusion**
- `contigs=''` `pdb='6MRR'` - noise all coordinates
- `contigs='A1-10'` `pdb='6MRR'` - keep first 10 positions fixed, noise the rest
- `contigs='A'` `pdb='1SSC'` - fix chain A, noise the rest

*hints and tips*
- `pdb=''` leave blank to get an upload prompt
- `contigs='50-100'` use dash to specify a range of lengths to sample from